# Module 01 — L'échelle de la douleur

**Formation Big Data — ANSD / Data Innovation Lab**

Nous avons ressenti le mur. Mesurons-le, puis servons-nous de ces mesures pour
répondre à la seule question qui compte vraiment quand on dimensionne un
traitement : **jusqu'où peut-on aller avec cette machine, et à partir de quand
faut-il changer d'outil ?**

Objectifs :

1. mesurer le temps de chaque opération à plusieurs volumes ;
2. en déduire la loi de croissance — le temps est-il proportionnel au volume ?
3. **extrapoler** aux volumes réels de la statistique publique sénégalaise ;
4. identifier le seuil de rupture de votre poste de travail.

C'est un exercice de dimensionnement, une compétence d'encadrant autant que de
praticien : savoir dire « ce traitement passera » ou « il ne passera pas », avant
de l'avoir lancé.

## 1. Protocole de mesure

In [ ]:
import gc
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

DOSSIER_DONNEES = Path("..") / "00-data"
FICHIER = DOSSIER_DONNEES / "individus.csv"

processus = psutil.Process(os.getpid())
memoire_mo = lambda: processus.memory_info().rss / 1024**2


def mesurer(nom, fonction):
    """Chronomètre une opération et renvoie sa durée en secondes."""
    gc.collect()
    depart = time.perf_counter()
    resultat = fonction()
    duree = time.perf_counter() - depart
    del resultat
    gc.collect()
    return duree


TAILLES = [100_000, 500_000, 1_000_000, 2_000_000]

print(f"Mémoire libre : {psutil.virtual_memory().available / 1024**3:.1f} Go")
print(f"Volumes testés : {', '.join(f'{n:,}'.replace(',', ' ') for n in TAILLES)}")

> **Si le temps manque**, passez `MESURER = False` dans la cellule
> suivante : des mesures de référence seront utilisées, et vous pourrez
> enchaîner directement sur l'analyse.

## 2. La campagne de mesures

Pour chaque volume, nous chronométrons cinq opérations : la lecture du fichier,
un filtre, une agrégation, un tri et une jointure.

In [ ]:
MESURER = True

# Mesures de référence (poste de démonstration), utilisées si MESURER = False
REFERENCE = pd.DataFrame({
    "lignes":   [100_000, 500_000, 1_000_000, 2_000_000],
    "lecture":  [1.477, 4.878, 7.317, 15.153],
    "filtre":   [0.151, 0.547, 0.504, 1.077],
    "groupby":  [0.007, 0.014, 0.038, 0.118],
    "tri":      [0.134, 0.321, 0.670, 2.079],
    "jointure": [0.178, 0.896, 1.449, 3.652],
})

In [ ]:
# Campagne de mesures : cinq opérations, à quatre volumes.
#
# Attention : chaque `lambda` capture la variable `df` du tour de boucle en
# cours — c'est bien ce que l'on veut ici, puisque `mesurer` l'appelle
# immédiatement.

if MESURER:
    lignes_resultats = []
    for n in TAILLES:
        print(f"Volume {n:,} …".replace(",", " "), end=" ", flush=True)
        df = pd.read_csv(FICHIER, nrows=n)
        reference = df[["id_individu", "nom"]].sample(frac=0.5, random_state=1)

        resultat = {
            "lignes": n,
            "lecture":  mesurer("lecture",
                                lambda: pd.read_csv(FICHIER, nrows=n)),
            "filtre":   mesurer("filtre",
                                lambda: df[df["age"] >= 15]),
            "groupby":  mesurer("groupby",
                                lambda: df.groupby("region")["age"].mean()),
            "tri":      mesurer("tri",
                                lambda: df.sort_values(["region", "age"])),
            "jointure": mesurer("jointure",
                                lambda: df.merge(reference, on="id_individu",
                                                 how="left")),
        }
        lignes_resultats.append(resultat)
        del df, reference
        gc.collect()
        print("terminé")

    mesures = pd.DataFrame(lignes_resultats)
else:
    mesures = REFERENCE.copy()

mesures.round(3)

## 3. La forme de la courbe

In [ ]:
operations = ["lecture", "filtre", "groupby", "tri", "jointure"]

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for op in operations:
    axes[0].plot(mesures["lignes"], mesures[op], marker="o", label=op)
axes[0].set_xlabel("nombre de lignes")
axes[0].set_ylabel("secondes")
axes[0].set_title("Temps par opération")
axes[0].legend()
axes[0].grid(alpha=0.3)

for op in operations:
    axes[1].plot(mesures["lignes"], mesures[op], marker="o", label=op)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("nombre de lignes (échelle log)")
axes[1].set_ylabel("secondes (échelle log)")
axes[1].set_title("Mêmes données, échelles logarithmiques")
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

En échelle logarithmique, une croissance **proportionnelle** au volume
apparaît comme une droite de pente 1 : deux fois plus de lignes, deux fois plus
de temps. Une pente plus forte signalerait une opération qui se dégrade plus
vite que le volume.

In [ ]:
# Facteur d'augmentation du temps, comparé au facteur d'augmentation du volume.
# Un rapport proche de 1 signale une croissance proportionnelle au volume.

facteur_volume = mesures["lignes"].iloc[-1] / mesures["lignes"].iloc[0]

comparaison = pd.DataFrame({
    "facteur_temps": [mesures[op].iloc[-1] / mesures[op].iloc[0]
                      for op in operations],
}, index=operations)
comparaison["facteur_volume"] = facteur_volume
comparaison["rapport"] = (comparaison["facteur_temps"] / facteur_volume).round(2)

print(f"Le volume a été multiplié par {facteur_volume:.0f}\n")
comparaison.round(2)

**Question 1.** Quelles opérations se comportent proportionnellement au
volume ? Y en a-t-il qui se dégradent plus vite ? Est-ce cohérent avec ce que
vous savez de ces opérations (un tri doit comparer les lignes entre elles, une
agrégation les parcourt une fois) ?

*Votre réponse :* …

**Question 2.** Sur les petits volumes, certaines opérations semblent
*meilleures* que proportionnelles. Quelle en est l'explication la plus probable ?

*Votre réponse :* …

## 4. Extrapolation : et à l'échelle du pays ?

Le temps croissant à peu près proportionnellement au volume, nous pouvons
extrapoler. C'est le calcul qu'un chef d'équipe doit savoir faire en réunion.

In [ ]:
# Coût unitaire, déduit de la plus grande mesure disponible
plus_grand = mesures.iloc[-1]
cout_par_million = {op: plus_grand[op] / (plus_grand["lignes"] / 1e6)
                    for op in operations}

# Mémoire observée au notebook 02 : ~320 octets par ligne une fois chargée
OCTETS_PAR_LIGNE = 320

pd.Series(cout_par_million).round(2).rename("secondes par million de lignes")

In [ ]:
# Projection aux volumes réels de la statistique publique.
#
# Temps : coût unitaire × volume, pour une chaîne
#         « lecture + filtre + groupby + tri ».
# Mémoire : 320 octets par ligne, et le DOUBLE pendant la lecture.

VOLUMES_REELS = {
    "Échantillon de travail":        1_000_000,
    "Recensement d'une région":      4_000_000,
    "Recensement national (RGPH)":  18_000_000,
    "Une année d'état civil":       60_000_000,
    "Un mois de données télécom":  500_000_000,
}

chaine = ["lecture", "filtre", "groupby", "tri"]
cout_chaine = sum(cout_par_million[op] for op in chaine)

projection = pd.DataFrame([
    {
        "jeu de données": libelle,
        "lignes": volume,
        "temps (min)": round(cout_chaine * volume / 1e6 / 60, 1),
        "mémoire (Go)": round(volume * OCTETS_PAR_LIGNE / 1024**3, 1),
        "pic lecture (Go)": round(2 * volume * OCTETS_PAR_LIGNE / 1024**3, 1),
    }
    for libelle, volume in VOLUMES_REELS.items()
]).set_index("jeu de données")

projection

In [ ]:
# Fourni : confrontation avec les ressources de VOTRE poste
memoire_totale_go = psutil.virtual_memory().total / 1024**3
memoire_libre_go = psutil.virtual_memory().available / 1024**3

print(f"Mémoire totale de ce poste  : {memoire_totale_go:.1f} Go")
print(f"Mémoire libre actuellement  : {memoire_libre_go:.1f} Go")

seuil_lignes = memoire_libre_go * 1024**3 / (OCTETS_PAR_LIGNE * 2)
print(f"\nVolume maximal chargeable   : "
      f"{seuil_lignes / 1e6:,.1f} millions de lignes".replace(",", " "))
print("(et encore : sans marge pour trier ou joindre ensuite)")

**Question 3.** Comparez ce seuil aux volumes du tableau de projection.
À partir de quel jeu de données votre poste décroche-t-il ?

*Votre réponse :* …

C'est ici que le raisonnement bascule. Sur la dimension du **temps**, la
dégradation est progressive : c'est deux fois plus long, puis dix fois plus
long, on peut lancer le traitement le soir et revenir le lendemain. Sur la
dimension de la **mémoire**, il n'y a pas de dégradation progressive : ça passe,
ou ça ne passe pas. La courbe ne se courbe pas — elle s'arrête.

Et lorsque le système tente de compenser en écrivant sur le disque (*swap*),
les temps ne sont plus multipliés par deux ou par trois, mais par cent.

## 5. Et un modèle statistique dans tout cela ?

Un algorithme d'apprentissage ne parcourt pas les données une fois : il les
parcourt à chaque itération, et souvent plusieurs fois par itération.

In [ ]:
# Fourni : ordre de grandeur pour un entraînement simple
ITERATIONS = 50   # ordre de grandeur pour une descente de gradient

for libelle, volume in [("Échantillon (1 M)", 1_000_000),
                        ("RGPH (18 M)", 18_000_000)]:
    parcours = cout_par_million["groupby"] * volume / 1e6
    total = parcours * ITERATIONS
    print(f"{libelle:<20} un parcours : {parcours:6.1f} s  |  "
          f"{ITERATIONS} itérations : {total / 60:7.1f} min")

**Question 4.** Ce calcul ne tient compte que du temps. Quel autre obstacle
rencontrerez-vous bien avant, sur le RGPH complet ?

*Votre réponse :* …

## 6. Synthèse du bloc

Complétez avec vos propres mesures :

| Constat | Votre mesure |
|---|---|
| Coût de lecture, par million de lignes | … s |
| Volume maximal chargeable sur ce poste | … millions de lignes |
| Temps projeté pour traiter le RGPH complet | … min |
| Mémoire nécessaire pour le RGPH complet | … Go |
| Opération la plus coûteuse | … |

**Ce que nous avons appris**

- Le temps de traitement croît à peu près proportionnellement au volume : il est
  donc **prévisible**, et se calcule avant de lancer un traitement.
- La mémoire, elle, impose un seuil net. En dessous, tout va bien ; au-dessus,
  rien ne fonctionne. Ce seuil se calcule aussi.
- Sur un poste de travail ordinaire, ce seuil se situe **en deçà des volumes
  courants de la statistique publique**.
- Aucune optimisation de code ne franchit ce seuil. Il faut changer de méthode :
  traiter les données **par morceaux**, et mobiliser **tous les cœurs**.


In [ ]:
# Sauvegarde des mesures, pour la comparaison avec Dask
Path("resultats").mkdir(exist_ok=True)
mesures.to_csv("resultats/mesures_pandas_par_volume.csv", index=False)
print("Mesures enregistrées : resultats/mesures_pandas_par_volume.csv")